In [1]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import scale, StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from tqdm import tqdm
import joblib
from sklearn.pipeline import make_pipeline
import utils

DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')

<module 'utils' from '/home/guoqiu/NAS/NAS4/guoqiu/Project_VMPFC/Connectivity_Predict_Activation_Testing/codes/utils/__init__.py'>

In [2]:
NAME = 'inhouse_BNA'
DATABASE_PATH = Path('../../Database/inhouse_dataset/')
SEED_LIST = SEED_DF['BNA'].dropna().to_list()  + SEED_DF['MDTB'].dropna().to_list()

SUBJECT_DF = pd.read_excel(DATA_PATH / 'subjects.xlsx')
SEED_DF = pd.read_excel(DATA_PATH / 'seeds.xlsx')

full_subject_list = SUBJECT_DF['subjects'].dropna().to_list()

In [3]:
try:
    X_dict = joblib.load(DATA_PATH / NAME / 'X.dict.pkl')
    y_dict = joblib.load(DATA_PATH / NAME / 'y.dict.pkl')
    mask_array_list = []
    for subject in full_subject_list:
        mask_file = DATABASE_PATH / "T1w" / subject / (subject + "_VMPFC2T1w.nii.gz")
        mask = nib.load(mask_file).get_fdata() > 0
        mask_array_list.append(mask[mask])

except FileNotFoundError:
    SC_array_list = []
    FC_array_list = []
    social_contrast_array_list = []
    valuation_contrast_array_list = []
    affective_contrast_array_list = []
    mask_array_list = []

    for subject in full_subject_list:
        print(subject, end=',')
        mask_file = DATABASE_PATH / "T1w" / subject / (subject + "_VMPFC2T1w.nii.gz")
        mask = nib.load(mask_file).get_fdata() > 0
        mask_array_list.append(mask[mask])
        subject_FC_array_list, subject_SC_array_list = [], []
        for seed in SEED_LIST:
            FC_file_path = DATABASE_PATH / "FC_T1" / subject / ("seeds_to_" + seed + ".nii.gz")
            FC_seed_array = nib.load(FC_file_path).get_fdata()[mask]
            subject_FC_array_list.append(FC_seed_array)
            # 注意这里有一处修改
            SC_file_path = DATABASE_PATH / "DTI_T1w" / subject / ("seeds_to_" + seed + ".nii.gz")
            SC_seed_array = nib.load(SC_file_path).get_fdata()[mask]
            subject_SC_array_list.append(SC_seed_array)
            # 类似的修改
        FC_array_list.append(np.array(subject_FC_array_list).T)
        SC_array_list.append(np.array(subject_SC_array_list).T)

        contrast_file_path = DATABASE_PATH / f"Task_T1w/tom_2nd/{subject}.gfeat/cope1.feat/stats/zstat1.nii.gz"
        contrast_array = nib.load(contrast_file_path).get_fdata()[mask].T
        social_contrast_array_list.append(contrast_array)

        contrast_file_path = DATABASE_PATH / f"Task_T1w/gam_2nd/{subject}.gfeat/cope1.feat/stats/zstat1.nii.gz"
        contrast_array = nib.load(contrast_file_path).get_fdata()[mask].T
        valuation_contrast_array_list.append(contrast_array)

        contrast_file_path = DATABASE_PATH / f"Task_T1w/ert_2nd/{subject}.gfeat/cope3.feat/stats/zstat1.nii.gz"
        contrast_array = nib.load(contrast_file_path).get_fdata()[mask].T
        affective_contrast_array_list.append(contrast_array)

    X_dict = dict(
    SC=np.vstack([scale(x) for x in SC_array_list]),
    FC=np.vstack([scale(x) for x in FC_array_list]))
    y_dict = dict(
        social=np.hstack([scale(x) for x in social_contrast_array_list]),
        valuation=np.hstack([scale(x) for x in valuation_contrast_array_list]),
        affective=np.hstack([scale(x) for x in affective_contrast_array_list]),)

    joblib.dump(X_dict, DATA_PATH / NAME / 'X.dict.pkl')
    joblib.dump(y_dict, DATA_PATH / NAME / 'y.dict.pkl')

In [4]:
group_list = np.hstack([np.ones_like(x) * (i + 1) for i, x in enumerate(mask_array_list)])
cv = LeaveOneGroupOut()
model = LinearRegression()

In [ ]:
PERMUTATION_NUM = 2000
for X_name, X_array in X_dict.items():
    for y_name, y_array in y_dict.items():
        name = f'{X_name}_{y_name}'

        include_subject_list = SUBJECT_DF[y_name].dropna().to_list()
        include_group_list = np.array([i + 1 for i, x in enumerate(full_subject_list) if x in include_subject_list])
        group_mask = [x in include_group_list for x in group_list]
        this_group_list, this_X_array, this_y_array = group_list[group_mask], X_array[group_mask], y_array[group_mask]

        # make predictions
        y_predict = cross_val_predict(estimator=model, X=this_X_array, y=this_y_array, cv=cv, groups=this_group_list, n_jobs=32, verbose=2)
        result_data_list = []

        r_dict = dict()
        r_dict['original'] = utils.get_r.calculate_prediction_r(this_group_list, this_y_array, y_predict)


        for permutation_i in tqdm(range(PERMUTATION_NUM)):
            # shuffle y for later permutation
            y_shuffle = np.zeros_like(this_y_array)
            for group_i in np.unique(this_group_list):
                group_mask = this_group_list == group_i
                y_shuffle[group_mask] = np.random.permutation(this_y_array[group_mask])
            # get permutation predict
            y_predict_permutation = cross_val_predict(estimator=model, X=this_X_array, y=y_shuffle, cv=cv, groups=this_group_list,
                                                      n_jobs=4, verbose=0)
            r_dict[f'permutation_{permutation_i}'] = utils.get_r.calculate_prediction_r(this_group_list, y_shuffle, y_predict_permutation)

        result_df= pd.DataFrame.from_dict(r_dict, orient='index')
        result_df = result_df.mean(axis=1)
        joblib.dump(result_df, DATA_PATH / NAME/ f'{name}_prediction.df.pkl')

        # final model's coefficient
        model.fit(this_X_array, this_y_array)
        joblib.dump(model, DATA_PATH / NAME/ f'{name}.LinearRegression.pkl')

Now Processing: SC_social


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done  15 out of  38 | elapsed:    5.6s remaining:    8.6s
[Parallel(n_jobs=32)]: Done  35 out of  38 | elapsed:    6.8s remaining:    0.6s
[Parallel(n_jobs=32)]: Done  38 out of  38 | elapsed:    7.0s finished
100%|██████████| 2000/2000 [3:25:14<00:00,  6.16s/it]  


Now Processing: SC_valuation


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   8 out of  35 | elapsed:    3.4s remaining:   11.5s
[Parallel(n_jobs=32)]: Done  26 out of  35 | elapsed:    3.8s remaining:    1.3s
[Parallel(n_jobs=32)]: Done  35 out of  35 | elapsed:    4.4s finished
100%|██████████| 2000/2000 [2:45:41<00:00,  4.97s/it]  


Now Processing: SC_affective


...

[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done  15 out of  38 | elapsed:    5.0s remaining:    7.7s
[Parallel(n_jobs=32)]: Done  35 out of  38 | elapsed:    5.3s remaining:    0.5s
[Parallel(n_jobs=32)]: Done  38 out of  38 | elapsed:    5.9s finished
100%|██████████| 2000/2000 [3:09:03<00:00,  5.67s/it]  


Now Processing: FC_valuation


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   8 out of  35 | elapsed:    3.2s remaining:   10.9s
[Parallel(n_jobs=32)]: Done  26 out of  35 | elapsed:    3.6s remaining:    1.2s
[Parallel(n_jobs=32)]: Done  35 out of  35 | elapsed:    4.2s finished
100%|██████████| 2000/2000 [2:40:40<00:00,  4.82s/it]  


Now Processing: FC_affective


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done  23 out of  42 | elapsed:    4.4s remaining:    3.7s
[Parallel(n_jobs=32)]: Done  42 out of  42 | elapsed:    5.4s finished
 48%|████▊     | 956/2000 [1:53:32<2:15:23,  7.78s/it]